In [0]:
# Persistent paths on Unity Catalog Volumes (survives serverless restarts)
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
CHROMA_DB_CANDIDATES = [
    "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test",
    "/Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test",
]

COLLECTION_NAME = "legal_knowledge"
PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"

BATCH_SIZE = 128
FORCE_REBUILD_CHROMA = False  # set True to rebuild collection from scratch


In [0]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import os
import json
import traceback
from datetime import datetime

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


In [0]:
try:
    gold_df = spark.read.format("delta").load(GOLD_PATH)
except Exception as e:
    raise RuntimeError(
        f"Unable to read gold chunks from {GOLD_PATH}. Make sure notebook 03 ran successfully. Error: {e}"
    )

required_cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name"]
missing_cols = [c for c in required_cols if c not in gold_df.columns]
if missing_cols:
    raise ValueError(f"Gold table is missing required columns: {missing_cols}")

gold_df = (
    gold_df.select(*required_cols)
    .dropna(subset=["chunk_id", "chunk_text"])
    .dropDuplicates(["chunk_id"])
)

gold_count = gold_df.count()
if gold_count == 0:
    raise ValueError("Gold dataset is empty after filtering. Cannot build embeddings.")

log(f"Gold chunks ready: {gold_count}")
gold_df.show(10, truncate=120)


[17:46:57] Gold chunks ready: 5194
+------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------+--------------+------------+----------------------------+
|                            chunk_id|                                                                                                              chunk_text|                act_name|section_number|    category|                   file_name|
+------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------+--------------+------------+----------------------------+
|000ae362-e61b-404f-82be-13a3a3542471|wing that A did not intend to harm the reputation of B. (f) A is sued by B for fraudulently representing to B that C ...|Indian Evidence Act 1872|          NULL|criminal_law|Indian Evidence Act_1872.pd

In [0]:
embedding_model = None
loaded_model_name = None

for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
    try:
        log(f"Loading embedding model: {model_name}")
        embedding_model = SentenceTransformer(model_name)
        _ = embedding_model.encode(["health check"], show_progress_bar=False)
        loaded_model_name = model_name
        log(f"Embedding model loaded: {model_name}")
        break
    except Exception as e:
        log(f"Failed to load {model_name}: {e}")

if embedding_model is None:
    raise RuntimeError(
        "Could not load any embedding model. Check internet access/Hugging Face access on cluster."
    )


[17:46:57] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[17:46:59] Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [0]:
os.makedirs("/Volumes/workspace/legal_data/vector_db_test", exist_ok=True)
os.makedirs("/Volumes/workspace/legal_data/chroma_db", exist_ok=True)

CHROMA_DB_PATH = None
client = None
collection = None
init_errors = []

for candidate in CHROMA_DB_CANDIDATES:
    try:
        os.makedirs(candidate, exist_ok=True)
        client = chromadb.PersistentClient(
            path=candidate,
            settings=Settings(anonymized_telemetry=False, allow_reset=True),
        )

        if FORCE_REBUILD_CHROMA:
            try:
                client.delete_collection(COLLECTION_NAME)
                log(f"Deleted existing collection at {candidate}")
            except Exception:
                pass

        collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )
        CHROMA_DB_PATH = candidate

        # Validate embedding dimension compatibility on reruns
        sample = embedding_model.encode(["dimension probe"], show_progress_bar=False)[0]
        expected_dim = len(sample)

        if collection.count() > 0:
            probe = collection.get(limit=1, include=["embeddings"])
            existing_embs = probe.get("embeddings") or []
            if existing_embs and len(existing_embs[0]) != expected_dim:
                log(
                    f"Existing collection dim ({len(existing_embs[0])}) != current model dim ({expected_dim}). Recreating collection."
                )
                client.delete_collection(COLLECTION_NAME)
                collection = client.get_or_create_collection(
                    name=COLLECTION_NAME,
                    metadata={"hnsw:space": "cosine"}
                )

        break
    except Exception as e:
        init_errors.append(f"{candidate}: {e}")

if collection is None:
    log("WARNING: Chroma initialization failed for all candidate paths.")
    for err in init_errors:
        log(f" - {err}")
    log("Continuing with persistent Delta embeddings only.")
else:
    log(f"Chroma collection ready at {CHROMA_DB_PATH}. Existing vectors: {collection.count()}")

manifest_path = "/Volumes/workspace/legal_data/vector_db_test/embedding_runtime_manifest.json"
manifest = {
    "embedding_delta_path": EMBEDDING_DELTA_PATH,
    "chroma_path": CHROMA_DB_PATH,
    "collection_name": COLLECTION_NAME,
    "embedding_model": loaded_model_name,
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log(f"Wrote runtime manifest to: {manifest_path}")


[17:47:00] WARNING: Chroma initialization failed for all candidate paths.
[17:47:00]  - /Volumes/workspace/legal_data/chroma_db/legal_knowledge_test: Could not connect to tenant default_tenant. Are you sure it exists?
[17:47:00]  - /Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test: Could not connect to tenant default_tenant. Are you sure it exists?
[17:47:00] Continuing with persistent Delta embeddings only.
[17:47:01] Wrote runtime manifest to: /Volumes/workspace/legal_data/vector_db_test/embedding_runtime_manifest.json


kum

In [0]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)


def build_metadata(row):
    return {
        "act_name": safe_str(row.act_name),
        "section": safe_str(row.section_number),
        "category": safe_str(row.category),
        "source": safe_str(row.file_name),
    }


In [0]:
records = []
processed = 0
chroma_upserts = 0
batch_rows = []


def flush_batch(rows):
    texts = []
    ids = []
    metadatas = []

    for r in rows:
        chunk_text = safe_str(r.chunk_text).strip()
        chunk_id = safe_str(r.chunk_id).strip()

        if not chunk_text or not chunk_id:
            continue

        texts.append(chunk_text)
        ids.append(chunk_id)
        metadatas.append(build_metadata(r))

    if not ids:
        return 0, []

    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()

    ts = datetime.utcnow().isoformat()
    batch_records = []
    for i in range(len(ids)):
        batch_records.append({
            "chunk_id": ids[i],
            "chunk_text": texts[i],
            "act_name": metadatas[i]["act_name"],
            "section_number": metadatas[i]["section"],
            "category": metadatas[i]["category"],
            "file_name": metadatas[i]["source"],
            "embedding": embeddings[i],
            "embedding_model": loaded_model_name,
            "embedding_dim": len(embeddings[i]),
            "updated_at": ts,
        })

    upserted = 0
    if collection is not None:
        try:
            collection.upsert(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas,
            )
            upserted = len(ids)
        except Exception as e:
            log(f"WARNING: Chroma upsert failed for current batch. Delta export will still continue. Error: {e}")

    return upserted, batch_records


for row in gold_df.toLocalIterator():
    batch_rows.append(row)

    if len(batch_rows) >= BATCH_SIZE:
        upserted, batch_records = flush_batch(batch_rows)
        chroma_upserts += upserted
        records.extend(batch_records)
        processed += len(batch_rows)
        log(f"Processed rows: {processed}")
        batch_rows = []

if batch_rows:
    upserted, batch_records = flush_batch(batch_rows)
    chroma_upserts += upserted
    records.extend(batch_records)
    processed += len(batch_rows)

if not records:
    raise RuntimeError("No embedding records were produced.")

embedding_df = spark.createDataFrame(records)
embedding_df = embedding_df.withColumn("updated_at", F.to_timestamp("updated_at"))

(
    embedding_df
    .dropDuplicates(["chunk_id"])
    .write
    .format("delta")
    .mode("overwrite")
    .save(EMBEDDING_DELTA_PATH)
)

log(f"Delta embedding export complete at: {EMBEDDING_DELTA_PATH}")
log(f"Rows written to Delta: {embedding_df.count()}")
log(f"Vectors upserted to Chroma in this run: {chroma_upserts}")
if collection is not None:
    log(f"Current Chroma vector count: {collection.count()}")


/home/spark-af03116e-9639-4f8a-872d-6f/.ipykernel/2545/command-6505964710941594-2763620553:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


[17:47:17] Processed rows: 128
[17:47:23] Processed rows: 256
[17:47:29] Processed rows: 384
[17:47:35] Processed rows: 512
[17:47:41] Processed rows: 640
[17:47:47] Processed rows: 768
[17:47:53] Processed rows: 896
[17:47:58] Processed rows: 1024
[17:48:04] Processed rows: 1152
[17:48:10] Processed rows: 1280
[17:48:15] Processed rows: 1408
[17:48:21] Processed rows: 1536
[17:48:27] Processed rows: 1664
[17:48:33] Processed rows: 1792
[17:48:38] Processed rows: 1920
[17:48:44] Processed rows: 2048
[17:48:50] Processed rows: 2176
[17:48:56] Processed rows: 2304
[17:49:03] Processed rows: 2432
[17:49:08] Processed rows: 2560
[17:49:14] Processed rows: 2688
[17:49:19] Processed rows: 2816
[17:49:25] Processed rows: 2944
[17:49:31] Processed rows: 3072
[17:49:37] Processed rows: 3200
[17:49:42] Processed rows: 3328
[17:49:48] Processed rows: 3456
[17:49:53] Processed rows: 3584
[17:49:59] Processed rows: 3712
[17:50:05] Processed rows: 3840
[17:50:11] Processed rows: 3968
[17:50:17] Proc

In [0]:
query = "What is the penalty for not wearing a helmet under Indian law?"

query_embedding = embedding_model.encode([query], show_progress_bar=False).tolist()

if collection is None:
    print("Chroma is unavailable in this session. Use 05 notebook, it will retrieve from Delta fallback.")
else:
    try:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=5,
        )

        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]

        print(f"Retrieved docs: {len(docs)}")
        for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
            print(f"\n--- Result {i} ---")
            print(meta)
            print(doc[:400])

    except Exception as e:
        print(f"Smoke test query failed: {e}")
        print("Embedding export is still available in Delta at EMBEDDING_DELTA_PATH.")


Chroma is unavailable in this session. Use 05 notebook, it will retrieve from Delta fallback.
